# Einfache Verwendung

## Importieren der benötigten Dateien und Klassen

Für die Verwendung der High-Level API benötigen wir:

* lims_interface.py aus dem Unterpaket "api"
* die Auswahl-Enums aus interaction_manager_factory.py aus dem Unterpaket "api"
* Zur Auswahl der verschiedenen Handler bei dynamischen Methoden das Auswahl-Enum "ConnectionType" aus interaction_manager.py
* Zur Auswahl eines RAG-Modus der Anwendung das Enum aus der Datei "settings.py" aus dem Unterpaket "utils"

Alternativ kann auch die gesamte Schnittstelle mit dem folgenden import geladen werden:
```
from llm_interaction_manager import *
```

Der vierte Import dient in diesem Beispiel rein der Codesicherheit, dass Nutzungstokens nicht im Code auftauchen sondern extern gespeichert sind

In [17]:
from llm_interaction_manager.api import lims_interface as api
from llm_interaction_manager.api.interaction_manager_factory import LLMEnum, PersistentEnum, VectorEnum
from llm_interaction_manager.core.interaction_manager import ConnectionType
from llm_interaction_manager.utils.settings import RAGMode

#Token aus Umgebungsvariable laden
import os

## Initialisieren der Schnittstelle, hier über High-Level API

Für diese Demo werden die folgenden Handler ausgewählt:

* LLMHandler: Langchain zur Verwendung von LLM als externer Dienst
* VectorHandler: ChromaDB
* PersistentHandler: MongoDB

In [18]:
api.initialize(llm=LLMEnum.LANGCHAIN, vector=VectorEnum.CHROMADB, persistent=PersistentEnum.MONGODB)

## Definieren der Verbindungsdaten und Aufbau der Verbindung

Die erste Zeile lädt das Nutzungstoken aus den Umgebungsvariablen des Systems

Dann werden drei **dict**s erstellt, welche die Verbindungsdaten zu den Endpunkten enthalten:

* Langchain benötigt die Auswahl eines LLM-Models und das Nutzungstoken für TogetherAI
* ChromaDB benötigt die Auswahl des Verbindungstypes, hier **PERSISTENT** und je nach Verbindungstyp weitere Parameter, hier den Pfad zur persistenten Speicherdatei
* MongoDB benötigt Verbindungsinformationen **Host** und **Port** sowie die Auswahl einer spezifischen Datenbank

Sind diese **dict**s erstellt, kann die Verbindung aufgebaut werden durch:
```
api.connect(ConnectionType, data)
```
Wobei **ConnectionType** einen spezifischen Handler-Typ definiert und data eines der gerade erstellten **dict**s des spezifischen Handlers ist.

In [19]:
#token aus Umgebungsvariable laden
token = os.getenv("together_ai_token")

#Verbindungsdaten
llm_data = {
    "model": "moonshotai/Kimi-K2-Instruct-0905",
    "token": token
}
vector_data = {
    "client_type": "PERSISTENT",
    "persistent_client_db_path": "D:/chroma"
}
persistent_data = {
    "host": "localhost",
    "port": 27017,
    "database": "promptDB"
}

api.connect(ConnectionType.LLM, llm_data)
api.connect(ConnectionType.VECTOR, vector_data)
api.connect(ConnectionType.PERSISTENT, persistent_data)

# Checking Connection
print(
    f"Connected: "
    f"LLM: {api.is_connected(ConnectionType.LLM)} "
    f"VECTOR: {api.is_connected(ConnectionType.VECTOR)} "
    f"PERSISTENT: {api.is_connected(ConnectionType.PERSISTENT)}"
)

Connected: LLM: True VECTOR: True PERSISTENT: True


# Weitere Anwendungsmöglichkeiten

## Einfache Anfrage an das LLM

Sind keine weiteren Konfigurationsschritte gewünscht kann eine einfache Anfrage an das LLM gesendet werden.
Da jede Anfrage im Kontext einer Konversation gespeichert wird und stattfinden muss, muss zuerst mit `start_conversation()` eine Konversation initiierrt werden.

Das eigentliche Prompt kann dann mit `send_prompt()` an das LLM gesendet werden.

In [20]:
api.start_conversation()
response = api.send_prompt("Einfaches Prompt")
print(response["content"])

Alles klar – Deployment-Test bestätigt.  
Einfaches Prompt verstanden und bereit zur Ausführung.


## Setzen eines Systemprompts

Werden die Prompts in einem bestimmten Kontext an das LLM gesendet kann dieser durch ein sogenanntes "Systemprompt" festgelegt werden. Dieses wird mit jedem Prompt ebenfalls an das LLM gesendet und beschreibt diesen Kontext.

Das Systemprompt befindet sich unter dem Schlüssel `default_system_prompt` in den Einstellungen, wird also mit der Methode `write_setting()` beschrieben.

In [21]:
api.write_setting("default_system_prompt", "Dies ist ein Deployment-Test")

## Hinzufügen von RAG-Daten

Soll das Prompt im Kontext von weiteren Daten ausgeführt werden, beispielsweise Daten, welche durch das LLM durchsucht oder zusammengefasst werden sollen, können diese als RAG-Daten an das Prompt angehängt werden.

Diese Daten können entweder persistent in den Einstellungen gespeichert werden, nur für die derzeitige Sitzung bestehen oder dynamisch aus vorherigen Konversationen geladen werden. Dieser Modus wird explizit durch `set_rag_mode()` gesetzt oder implizit durch das hinzufügen neuer RAG-Daten durch `set_rag_data()`.

In [26]:
rag_data = {"doc1":"Der Himmel ist grün", "doc2":"Die Erde ist rot"}

api.set_rag_data(rag_data, volatile=False)
rag_response = api.send_prompt("Welche Farbe hat der Himmel in den Daten?")
print(rag_response["content"])

api.set_rag_mode(RAGMode.DYNAMIC)
#in dieser Antwort werden KEINE RAG-Daten vom Nutzer direkt verwendet, sondern aus vorherigen Ergebnissen dynamisch bezogen
dyn_response = api.send_prompt("Welche Farbe hat die Erde?")
print(dyn_response["content"])

In den Daten ist der Himmel grün.
START
['PROMPT: SYSTEM PROMPT: Dies ist ein Deployment-Test PROMPT: Welche Farbe hat die Erde?\nRESPONSE: Antwort: Laut den Daten ist die Erde **nicht explizit erwähnt** – es gibt keine Information über ihre Farbe.', 'PROMPT: SYSTEM PROMPT: Dies ist ein Deployment-Test PROMPT: Welche Farbe hat die Erde?\nRESPONSE: Antwort: Laut den Daten ist die Erde **nicht explizit erwähnt** – es gibt keine Information über ihre Farbe.', 'PROMPT: SYSTEM PROMPT: Dies ist ein Deployment-Test PROMPT: Welche Farbe hat die Erde?\nRESPONSE: Laut den Daten ist die Erde **nicht explizit erwähnt** – es gibt keine Information über ihre Farbe.']
END
{'message_id': 'msg_8971-985c-48a1-8d24-aea564973a65', 'prompt': 'SYSTEM PROMPT: Dies ist ein Deployment-Test PROMPT: Welche Farbe hat die Erde?', 'content': 'Antwort: Laut den Daten ist die Erde **nicht explizit erwähnt** – es gibt keine Information über ihre Farbe.', 'comment': '', 'metadata': {'metadata': {'token_usage': {'comple

## Ähnlichkeitssuche in der Vektordatenbank

Alternativ zur Kommunikation mit einem LLM kann eine Eingabe auch zur Ähnlichkeitssuche auf der Vektordatenbank vorheriger Ergebnisse durchgeführt werden.

Standardmäßig werden Daten, welche von der Anwendung generiert werden in der Vektortabelle **lims_embeddings** gespeichert.

In [ ]:
print(api.nearest_search_vector("Dokumente mit Aussagen über den Himmel", 1, "lims_embeddings"))